# When crop prices rise, do tropical forests fall?

> **To write (Andrew):** a subtitle (one line) and a hook of 3-4 sentences for a general reader. Ideas: the 2024 cocoa price
> spike (cocoa more than doubled); tropical forests lost every year; the question of the article: do world
> crop prices drive deforestation, and can protection stop it? Say in one sentence what we find (prices
> explain about 44% of predicted deforestation since 2001, the effect is regional, strict protection
> reduces it by about two thirds).

*Authors: Andrew Eberhardt, David ... | Data Science Project, 2026. Technical details are in the Appendix and in notebooks 3 and 4.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("default")
plt.rcParams["figure.facecolor"] = "white"

# All data are read from GitHub. Switch BRANCH to "main" once the branch is merged.
BRANCH = "andrew-cleanup-berman"
BASE = f"https://raw.githubusercontent.com/AndrewEberhardt/DataScienceProject/{BRANCH}/data/"
RES = BASE + "processed/results/"           # results of notebook 4 (9 km models precomputed)

BERMAN_BETA = 1.27
SRC_FOREST = "Hansen et al. GFC v1.13"
SRC_GAEZ = "FAO/IIASA GAEZ v4"
SRC_PRICE = "World Bank Pink Sheet"
summary = pd.read_csv(RES + "summary.csv").set_index("model")

## 1. A decade and a half of crop price shocks

> **To write (Andrew):** 2 short paragraphs. Points: prices of tropical crops are set on world markets; they moved a lot
> since 2000 (food crisis 2007-2008, commodity boom 2011, 2021-2022 spike, cocoa and coffee records in
> 2024-2025); a farmer at the forest edge sees these prices; read the figure below (which crop moved most).

In [ ]:
prices = pd.read_csv(BASE + "processed/commodity_prices_annual.csv")
SERIES = {"Cocoa": ["Cocoa"], "Coffee": ["Coffee, Arabica", "Coffee, Robusta"], "Palm oil": ["Palm oil"],
          "Soybeans": ["Soybeans"]}
P = pd.DataFrame({k: prices[prices.commodity.isin(v)].groupby("year").price_real.mean() for k, v in SERIES.items()})
P = (100 * P / P.loc[2000]).loc[2000:2025]

fig, ax = plt.subplots(figsize=(9, 5))
for crop in P:
    ax.plot(P.index, P[crop], lw=2, label=crop)
ax.axhline(100, color="grey", lw=0.6)
ax.set_xlabel("Year")
ax.set_ylabel("Real world price (2000 = 100)")
ax.set_title("World prices of four forest-risk crops, 2000-2025")
ax.legend()
fig.text(0.01, -0.03, f"Source: {SRC_PRICE}, real prices (coffee: mean of Arabica and Robusta).", fontsize=9)
plt.show()

## 2. Not every forest feels the same price

> **To write (Andrew):** explain the idea of the price index without equations (the formula is in the Appendix). Points:
> world prices are the same everywhere, but a forest cell where cocoa grows well is exposed to the cocoa
> price, a cell where soybeans grow well to the soybean price; we weight each crop by how suitable the cell
> is (FAO GAEZ data); we use 511,427 cells of 9 km in 62 tropical countries, 6 times finer than Berman et al.
> (55 km); we compare cells of the same country in the same year, so national policies and crises cancel out.
> Comment the maps: cocoa and oil palm in humid forests (Amazon, Congo, Southeast Asia), soybeans at the
> edges (southern Brazil, India, Australia).

In [ ]:
CROPS = ["banana", "cocoa", "coconut", "coffee", "cotton", "maize", "oil palm", "rice", "soybean",
         "sugarcane", "tea", "tobacco", "wheat"]                  # the 13 crops of the main index
cells = pd.read_csv(BASE + "processed/grid_cells.csv.gz", usecols=["cell_id", "row", "col"])
suit = pd.read_csv(BASE + "processed/grid_suitability.csv.gz")
suit.columns = ["cell_id"] + CROPS + ["barley", "sorghum"]
S = cells[["cell_id"]].merge(suit, on="cell_id", how="left")[CROPS].to_numpy(float)
tot = S.sum(axis=1, keepdims=True)
alpha = pd.DataFrame(np.divide(S, tot, out=np.full_like(S, np.nan), where=tot > 0), columns=CROPS)

r0, r1, c0, c1 = cells.row.min(), cells.row.max() + 1, cells.col.min(), cells.col.max() + 1
extent = [-180 + c0 / 12, -180 + c1 / 12, 90 - r1 / 12, 90 - r0 / 12]
vmax = np.nanpercentile(alpha.to_numpy(), 99)

fig, axes = plt.subplots(2, 2, figsize=(18, 9))
for ax, crop in zip(axes.flat, ["cocoa", "oil palm", "soybean", "coffee"]):
    g = np.full((r1 - r0, c1 - c0), np.nan)
    g[cells.row - r0, cells.col - c0] = alpha[crop]
    im = ax.imshow(g, extent=extent, cmap="YlGn", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_facecolor("whitesmoke")
    ax.set_title(crop)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
fig.suptitle("Where four forest-risk crops weigh in the price index of each 9 km forest cell", fontsize=14)
b = 1.1 / 9
fig.tight_layout(rect=(0, b, 1, 1))
cax = fig.add_axes([0.3, 0.65 * b, 0.4, 0.15 / 9])
fig.colorbar(im, cax=cax, orientation="horizontal", extend="max",
             label="Weight of the crop in the cell's price index (relative suitability)")
fig.text(0.01, 0.0, f"Source: {SRC_GAEZ}, {SRC_FOREST}. Grey: no forest in 2000.", fontsize=9)
plt.show()

## 3. Higher prices, more deforestation

> **To write (Andrew):** the main result in plain words. Numbers to cite: a 10% rise in a cell's crop price index raises its
> deforestation by about 10% (coefficient 0.96, printed below); price changes since 2001 are responsible for
> about 44% of predicted deforestation over 2001-2025 (Berman et al.: 35% over 2001-2018); the effect is
> larger after 2018 (+0.31). Explain the permutation test in one sentence (if we give each crop the price of
> another crop, we almost never find an effect this large: p = 0.006).

In [ ]:
contrib = pd.read_csv(RES + "contribution.csv").set_index("model")
inference = pd.read_csv(RES + "inference.csv").set_index("model")
main = "MAIN: 9 km, 2001-2025, 13 crops"
print(f"Main coefficient: {summary.loc[main, 'Estimate']:.2f} (95% CI {summary.loc[main, '2.5%']:.2f} to "
      f"{summary.loc[main, '97.5%']:.2f}), extra effect after 2018: "
      f"{summary.loc['9 km, 2001-2025: extra effect after 2018', 'Estimate']:+.2f}")
print(f"Share of predicted deforestation due to price changes since 2001: {contrib.loc[main, 'share_due_to_prices']:.0%}")
print(f"Permutation p-value: {inference.loc[main, 'permutation p']:.3f}")

In [ ]:
perm = pd.read_csv(RES + "permutation_9km.csv")[main]
b0, draws = perm.iloc[0], perm.iloc[1:]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(draws, bins=40, color="lightgrey", edgecolor="grey", label="1,000 placebo indices (prices shuffled across crops)")
ax.axvline(b0, color="tab:red", lw=2.5, label=f"True price index: {b0:.2f}")
ax.set_xlabel("Estimated effect of the log price index on deforestation")
ax.set_ylabel("Number of placebo indices")
ax.set_title("The true prices beat almost every placebo (9 km cells, 2001-2025)")
ax.legend()
fig.text(0.01, -0.03, f"Source: {SRC_FOREST}, {SRC_GAEZ}, {SRC_PRICE}; own estimates.", fontsize=9)
plt.show()

## 4. Checking the original study

> **To write (Andrew):** what replicating means and what we found. Points: on Berman et al.'s own 55 km cells and period we
> find 1.97 instead of their 1.27; with their own files we get exactly 1.27; the forest data are identical
> (their outcome or ours gives the same result); the whole gap comes from the price index (two price series,
> banana and sugar, and an older version of the FAO data). Lesson for the reader: the sign and significance
> are solid, the exact size depends on choices that papers rarely discuss.

In [ ]:
comp = pd.read_csv(BASE + "processed/berman_comparison.csv")
keep = ["Berman et al. data and index", "Our outcome, Berman et al. index", "Berman et al. outcome, our index",
        "Our outcome, our index", "Our outcome, our suitability, Berman et al. prices",
        "Our outcome, Berman et al. suitability, our prices"]
r = comp.set_index("model").loc[keep[::-1]]
fig, ax = plt.subplots(figsize=(9, 4.5))
y = np.arange(len(r))
ax.errorbar(r.beta, y, xerr=1.96 * r.se, fmt="o", capsize=3, label="Estimate (95% CI)")
ax.axvline(BERMAN_BETA, color="tab:red", ls="--", lw=1, label="Published result: 1.27")
ax.set_yticks(y)
ax.set_yticklabels(r.index)
ax.set_xlabel("Effect of the log price index on deforestation (55 km cells, 2001-2018)")
ax.set_title("Same forests, different price index: where the gap with Berman et al. comes from")
ax.legend(loc="upper right", fontsize=8)
fig.text(0.01, -0.03, f"Source: Berman et al. (2023) replication files (Zenodo 7044013), {SRC_FOREST}, "
         f"{SRC_GAEZ}, {SRC_PRICE}; own estimates.", fontsize=9)
plt.show()

## 5. The price that matters is the region's, not the plot's

> **To write (Andrew):** the neighbour result. Points: we add the price index of the land around each cell, in rings up to
> 110 km; the effect moves to the ring at 60-110 km and the cell's own effect falls from 0.97 to about 0.16;
> interpretation: deforestation responds to the crop mix of the whole region (roads, mills, labour, traders),
> or the suitability of a single 9 km cell is a noisy measure; consequence: finer maps do not capture the
> mechanism better, and policies must think at the regional scale.

In [ ]:
nb = pd.read_csv(RES + "neighbours.csv").set_index("model")
fig, ax = plt.subplots(figsize=(9, 4.5))
y = np.arange(len(nb))[::-1]
ax.errorbar(nb.total, y, xerr=1.96 * nb["total SE"], fmt="o", capsize=3, label="Total effect, cell + neighbours (95% CI)")
ax.scatter(nb["lnP (beta)"], y, marker="x", s=60, color="tab:orange", label="Effect of the cell's own index")
ax.set_yticks(y)
ax.set_yticklabels(nb.index)
ax.set_xlabel("Effect of the log price index on deforestation")
ax.set_title("Adding the prices of the surrounding land (rings up to 110 km)")
ax.legend(fontsize=8, loc="upper left")
fig.text(0.01, -0.03, f"Source: {SRC_FOREST}, {SRC_GAEZ}, {SRC_PRICE}; own estimates.", fontsize=9)
plt.show()

## 6. Can protection stop it?

> **To write (Andrew):** the protected areas result and its limit. Points: protected forest cells lost 7.3% of their forest
> over 2001-2025, the others 14.2%; comparing cells with the same forest cover, a fully protected cell
> responds about 30% less to prices, and about 65% less under strict protection (IUCN I-IV); but areas
> protected after 2000 were created where pressure was already high (+0.26): the location of parks is not
> random, so this is a strong correlation, not proof that parks cause the difference.

In [ ]:
pa = pd.read_csv(RES + "protected_areas.csv")
fig, ax = plt.subplots(figsize=(9, 4.5))
y = np.arange(len(pa))[::-1]
ax.errorbar(pa.Estimate, y, xerr=1.96 * pa["Std. Error"], fmt="o", capsize=3, label="Interaction (95% CI)")
ax.axvline(0, color="grey", lw=0.6)
ax.set_yticks(y)
ax.set_yticklabels(pa.model)
ax.set_xlabel("Change in the price effect for a fully protected cell (main effect about 0.96)")
ax.set_title("Protected cells respond less to crop prices")
ax.legend(fontsize=8)
fig.text(0.01, -0.03, f"Source: WDPA (UNEP-WCMC and IUCN, Nov. 2026), {SRC_FOREST}, {SRC_GAEZ}, "
         f"{SRC_PRICE}; own estimates.", fontsize=9)
plt.show()

## 7. Why beef is not in the index

> **To write (Andrew):** short section from notebook 5 (David, beef exports, task 5 not done yet). Point: world beef prices
> reach local markets only in exporting countries (0 of 21 countries show a correlation above 0.5 between
> world and local prices), so a world price index cannot capture cattle-driven deforestation.

## 8. What this means

> **To write (Andrew):** conclusion of 2 paragraphs. Points: what we learned (prices matter, about 44% of predicted
> deforestation, regional scale, strict protection works best); limits (correlation for protected areas,
> the size of the effect depends on how the index is built, beef not covered); one sentence for policy
> (price booms are predictable warning signals; protection and regional policies matter most when prices
> rise).

In [ ]:
# Word count of the article (markdown cells before the Appendix; the limit is 3,000 words)
import json, re
nbk = json.load(open("6-Article.ipynb"))
words = 0
for c in nbk["cells"]:
    s = "".join(c["source"])
    if s.startswith("# Appendix"):
        break
    if c["cell_type"] == "markdown" and "To write (Andrew)" not in s:
        words += len(re.findall(r"\w+", s))
print(f"Article length: {words:,} words (limit 3,000)")

# Appendix: technical details

### A.1 Data

| Data | Source | Use |
|---|---|---|
| Forest loss 2001-2025, tree cover 2000 | Hansen et al. Global Forest Change v1.13 | Outcome: pixels lost per cell and year |
| Crop suitability, 15 crops | FAO/IIASA GAEZ v4, high input, rainfed | Weights of the price index |
| Real world crop prices | World Bank Commodity Price Data (Pink Sheet) | Shifts of the price index |
| Protected areas | WDPA, UNEP-WCMC and IUCN, November 2026 | Section 6 |
| Replication files | Berman et al. (2023), Zenodo 7044013 | Section 4 |

Full log of sources: `docs/data_log.md`. Processing: `scripts/` (run once, locally) and notebooks 3 and 4.

### A.2 Model (Berman et al. 2023, eq. 3)
```
asinh(Defor[c,t]) = beta * ln(Price[c,t]) + eta[c] + nu[country,t] + e[c,t]
Price[c,t] = sum_i alpha[i,c] * P[i,t],   alpha[i,c] = S[i,c] / sum_j S[j,c]
```

> **To write (Andrew):** explain the model in 4-5 sentences: what c, t, eta and nu are; why asinh (zeros); why country x year
> fixed effects; standard errors clustered by 0.5 degree block (plus country and Conley in notebook 4);
> the permutation test (Adao, Kolesar and Morales 2019).

### A.3 All estimates

In [ ]:
summary[["Estimate", "Std. Error", "2.5%", "97.5%", "N"]].round(3)

### A.4 Inference

In [ ]:
inference.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 0.42 * len(summary) + 1))
y = np.arange(len(summary))[::-1]
ax.errorbar(summary.Estimate, y, xerr=1.96 * summary["Std. Error"], fmt="o", capsize=3, label="Estimate (95% CI)")
ax.axvline(0, color="grey", lw=0.6)
ax.axvline(BERMAN_BETA, color="tab:red", ls="--", lw=1, label="Berman et al. (2023): 1.27")
ax.set_yticks(y)
ax.set_yticklabels(summary.index)
ax.set_xlabel("Effect of the log crop price index on asinh(deforested pixels), 95% CI")
ax.set_title("All specifications")
ax.legend(loc="lower right")
fig.text(0.01, -0.02, f"Source: {SRC_FOREST}, {SRC_GAEZ}, {SRC_PRICE}; own estimates.", fontsize=9)
plt.show()

### A.5 Reproducibility
Notebook 3 builds the price index; notebook 4 runs all models (by default it reads the 9 km results from
`data/processed/results/`; set `RECOMPUTE = True` to rerun them, about 15 minutes and 10 GB of RAM).
Heavy preprocessing (Hansen tiles, WDPA polygons, Berman et al.'s files) is in `scripts/`.

**References.** Berman, N., Couttenier, M., Leblois, A. and Soubeyran, R. (2023), Crop prices and deforestation
in the tropics, *Journal of Environmental Economics and Management*. Adao, R., Kolesar, M. and Morales, E.
(2019), Shift-share designs: theory and inference, *Quarterly Journal of Economics*. Hansen, M. et al. (2013),
High-resolution global maps of 21st-century forest cover change, *Science*.